In [1]:
!pip install openai langchain_core langchain_openai

# 라이브러리 불러오기
import openai
from typing import List


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# LM Studio 로컬 서버에 연결
client = openai.OpenAI(
    base_url="http://host.docker.internal:12345/v1",  # 로컬 LM Studio 주소 + /v1
    api_key="lm-studio",                      # 임의의 문자열 (검증 안 함)
)

In [6]:
# "안녕하세요!" 메시지를 보내고 응답을 받음
response = client.chat.completions.create(
    model="unsloth/gemma-4-e2b-it",                      # LM Studio에 로드된 모델명
    messages=[{"role": "user", "content": "안녕하세요!"}],
)
response.choices[0].message.content

'안녕하세요! 저는 Google에서 훈련한 대규모 언어 모델입니다.\n\n무엇을 도와드릴까요? 궁금한 점이 있으시거나, 도움이 필요하시면 언제든지 말씀해주세요! 😊'

In [7]:
# 요청에 사용할 프롬프트 템플릿 정의
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요."

# 메시지를 보내고 모델의 응답을 받는 함수
def call_chat_model(messages: List[dict]):
    response = client.chat.completions.create(
        model="unsloth/gemma-4-e2b-it",
        messages=messages,
    )
    return response.choices[0].message.content

# 주어진 주제에 따라 설명을 요청하는 함수
def invoke_chain(topic: str):
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

# "더블딥" 주제로 설명 요청
invoke_chain("더블딥")

"## 주제 더블딥(Topic Double Dip)에 대한 짧은 설명\n\n**주제 더블딥(Topic Double Dip)**은 **하나의 주제나 아이디어를 두 번 이상, 혹은 매우 밀접하게 반복하여 다루는 행위**를 의미합니다.\n\n쉽게 말해, **같은 이야기를 너무 자주, 혹은 지나치게 깊이 파고들어 지루하거나 불필요하게 느껴질 수 있는 상황**을 말합니다.\n\n### 주요 특징:\n\n* **반복성:** 동일한 핵심 아이디어나 논점을 여러 번 반복합니다.\n* **깊이의 문제:** 한 주제를 너무 많이 다루다 보면, 피상적인 설명에 머물거나 깊이 있는 분석 없이 겉핥기식으로 느껴질 수 있습니다.\n* **청중/독자의 피로도:** 듣는 사람이나 읽는 사람이 같은 내용을 반복해서 들으면 지루함이나 피로감을 느낄 수 있습니다.\n\n### 언제 사용되나요?\n\n* **발표나 글쓰기에서:** 핵심 메시지를 강조하기 위해 반복할 수는 있지만, 너무 과하면 독자를 지치게 합니다.\n* **토론이나 대화에서:** 논점을 명확히 하기 위해 반복할 수는 있으나, 새로운 관점이나 다른 측면을 추가해야 할 때가 있습니다.\n\n**요약하자면, 주제 더블딥은 '같은 것을 너무 많이 말하는 것'으로, 효과적인 소통을 위해서는 적절한 반복과 새로운 시각의 균형이 중요합니다.**"

In [8]:
invoke_chain("llm모델의 도구 호출의 정확성 보장")

'## LLM 모델의 도구 호출 정확성 보장에 대한 설명\n\n**LLM(거대 언어 모델)이 외부 도구를 호출할 때 발생하는 \'도구 호출의 정확성\'은 모델의 신뢰성과 실제 작업 수행 능력에 매우 중요한 요소입니다.** 이는 모델이 사용자의 의도를 정확히 파악하여 **적절한 도구를 선택하고, 올바른 매개변수(인수)를 사용하여 해당 도구를 실행**하도록 보장하는 것을 의미합니다.\n\n### 핵심적인 문제점\n\nLLM은 텍스트 기반의 패턴 학습을 통해 작동하기 때문에 다음과 같은 이유로 정확성 문제가 발생할 수 있습니다.\n\n1. **모호한 사용자 입력:** 사용자의 질문이 모호하거나 여러 가지 해석이 가능할 때, 모델이 가장 적절한 도구를 선택하지 못하고 잘못된 도구를 호출할 수 있습니다.\n2. **복잡한 추론 요구:** 도구 호출을 위해서는 단순한 텍스트 이해를 넘어선 복잡한 논리적 추론(예: "먼저 A를 검색하고, 그 결과를 바탕으로 B를 계산하라")이 필요하며, 이 과정에서 오류가 발생할 수 있습니다.\n3. **도구 설명의 불완전성:** 제공된 도구(API)에 대한 설명(Docstring)이 불충분하거나 부정확하면, 모델은 해당 도구가 실제로 수행할 수 있는 작업을 오해하고 잘못된 인수를 전달하게 됩니다.\n\n### 정확성 보장 전략\n\n도구 호출의 정확성을 높이기 위해 다음과 같은 접근 방식들이 사용됩니다.\n\n1. **프롬프트 엔지니어링 (Prompt Engineering):**\n    * **명확한 지침 제공:** 모델에게 도구 사용 규칙, 각 도구의 기능, 그리고 언제 어떤 도구를 사용해야 하는지에 대한 구체적이고 명확한 지침을 제공합니다.\n    * **Few-shot 예시:** 올바른 도구 호출과 그에 따른 결과의 예시를 함께 제공하여 모델이 원하는 출력 형식을 학습하도록 돕습니다.\n\n2. **모델 자체의 개선 (Fine-tuning & RAG):**\n    * **도메인 특화 파인튜닝:** 특정 작업이나 도구 사용 시나